# Colorectal Cancer Histology Classification using PathMNIST

This notebook will cover data preparation, model fine-tuning, evaluation, and interpretability using the PathMNIST dataset.

This dataset comes in [MedMNIST](https://github.com/MedMNIST/MedMNIST/), a large-scale MNIST-like collection of standardised biomedical images, including 12 datasets for 2D and 6 datasets for 3D. All images are standardised into multiple size options with the corresponding classification labels, so that no background knowledge is required for users.


PathMNIST is based on a study for predicting survival from colorectal cancer histology slides, and we will use non-overlapping 28x28 image patches extracted from Hematoxylin and Eosin (H&E) stained histological images.

You will develop a multi-class classification task, with the following classes:


* Adipose
* Background
* Debris
* Lymphocytes
* Mucus
* Smooth muscle
* Normal colon mucosa
* Cancer-associated stroma
* Colorectal adenocarcinoma epithelium





Let's start by installing and importing the right packages

In [ ]:
# Setup Environment
!pip install medmnist grad-cam

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import medmnist
from medmnist import INFO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Exercise 1: Data Loading & Preprocessing

**Task:** Load the PathMNIST dataset and prepare PyTorch DataLoaders.

**Pointers:**
* Use `medmnist.INFO['pathmnist']` to retrieve dataset metadata and the corresponding Python class.
* MedMNIST images are 28x28. Pre-trained models like ResNet expect 224x224, so you should apply a resize transformation.
* Convert to tensor and normalize using standard ImageNet mean `[0.485, 0.456, 0.406]` and standard deviation `[0.229, 0.224, 0.225]`.
* Instantiate training, validation, and testing datasets and their respective DataLoaders.

**Suggestion:**
* Start with only 10% of the training and validation sets for training to be faster.

In [ ]:
data_flag = 'pathmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

# Define transforms
data_transform = transforms.Compose([
    # Resize, ToTensor, Normalize
])

# Load datasets
# train_dataset = DataClass(split=...
# val_dataset = ...
# test_dataset = ...

# Subsample 10% of the data to accelerate training
# train_dataset, _ = random_split(train_dataset, [0.1, 0.9])
# val_dataset, _ = random_split(val_dataset, [0.1, 0.9])

# Create DataLoaders
# train_loader = ...
# val_loader = ...
# test_loader = ...

# Exercise 2: Model fine-tuning

**Task:** Initialise a pre-trained ResNet18 model, and fine-tune it on PathMNIST

**Pointers:**
* Use `torchvision.models.resnet18(pretrained=True)`.
* Replace the final fully connected layer to match the 9 classes of PathMNIST.

**Suggestions:**
* Start with optim.Adam with a learning rate of 1e-3.
* Train only for 3 epochs on a first try such that you can more quickly do all the notebook's exercises in today's session.
* Don't forget to apply a validation step per epoch to keep track of potential overfitting.


In [ ]:
# Initialize model
# model = ...

# Loss and Optimizer
# criterion = ...
# optimizer = ...

# Training Loop
num_epochs = 3
# for epoch in range(num_epochs):
#     # Implement training and validation steps

# Exercise 3: Performance evaluation

**Task:** Evaluate the trained model on the test set

**Pointers:**
* Include at least the Macro F1 score, such that we can use that metric for comparison with other people and see who is able to get the highest performance.
* Beyond the usual metrics, also plot a confusion matrix.




In [ ]:
# Collect predictions and true labels
y_true = []
y_pred = []

# model.eval()
# with torch.no_grad():
#     # Iterate over test_loader and populate y_true, y_pred

# Compute metrics
# acc = ...
# f1 = ...
# cm = ...

# Exercise 4: Visualisation and Interpretability

**Task:** Apply Grad-CAM to patches from the test set to visually check predictions and try to interpret model's predictions by visualising attention heatmaps.

**Pointers:**
* Target the last convolutional layer of ResNet18: `target_layers = [model.layer4[-1]]`.
* Don't forget to "un-normalize" the tensor image for visualization: `img_viz = img_tensor.permute(1, 2, 0).numpy() * std + mean`.
* Include the ground truth and predicted labels information such that you can see if you are seeing a heatmap of a correct or wrong prediction.


**Suggestion:**
* Use `pytorch_grad_cam`'s `show_cam_on_image(img_viz, grayscale_cam, use_rgb=True)` (already imported) to make it easier to overlay the heatmap on the input image.


In [ ]:
# Select 5 samples as a starting point
# subset_indices = range(5)
# subset = torch.utils.data.Subset(test_dataset, subset_indices)

# target_layers = [model.layer4[-1]]
# cam = GradCAM(model=model, target_layers=target_layers)

# for img, label in subset:
#     # Prepare input tensor, generate CAM, unnormalize image, plot
#     pass

# Exercise 5: Improving performance

**Task:** Now that you have all the code needed to train and run inference on PathMNIST, try to improve the performance you get on the test set.


**Suggestions:**

*(Notice that some of them will likely have a different time/impact balance)*
* Increase the proportion of the training/validation data used and extend the number of epochs used for training.
* Replace ResNet18 with other architectures (eg `efficientnet_b0` or `vit_b_16`)  and analyse how this choice impacts convergence speed and Grad-CAM spatial localisation.
* Change the transfer learning strategy to freeze all feature layers (`param.requires_grad = False`) but the final one, such that you fine-tune only the final classification head.
* Introduce stain augmentation techniques during training.